# Chapter 4
Use the `base` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
I = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

CNOT_01 = np.array(
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ],
    dtype=complex,
)


def kron_all(ops):
    output = ops[0]
    for op in ops[1:]:
        output = np.kron(output, op)
    return output


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2),  np.cos(theta / 2)],
        ],
        dtype=complex,
    )


def rz(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.exp(-0.5j * theta), 0.0],
            [0.0, np.exp(0.5j * theta)],
        ],
        dtype=complex,
    )


def angle_encoding(x: np.ndarray) -> np.ndarray:
    return kron_all([ry(x[0]), ry(x[1])])


def variational_block(params: np.ndarray) -> np.ndarray:
    theta0, theta1, theta2, theta3 = params
    local_z = kron_all([rz(theta0), rz(theta1)])
    local_y = kron_all([ry(theta2), ry(theta3)])
    return CNOT_01 @ local_y @ local_z


def vqc_score(
    x: np.ndarray, params: np.ndarray, observable=None
) -> float:
    """Return an expectation, defaulting to Z on the first qubit."""
    state = kron_all([ket0, ket0])
    state = angle_encoding(x) @ state
    state = variational_block(params) @ state
    if observable is None:
        observable = kron_all([Z, I])
    return float(np.vdot(state, observable @ state).real)


params = np.array([0.3, -0.4, 0.7, 0.2])
for x in [
    np.array([0.1, 0.2]),
    np.array([1.0, -0.3]),
    np.array([-0.8, 0.6]),
]:
    score = vqc_score(x, params)
    print(x, "score=", score, "class=", int(score >= 0.0))


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
keep = iris.target < 2
X_raw = iris.data[keep, 2:4]
y = 2 * iris.target[keep] - 1
X_dev, X_test_raw, y_dev, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=7, stratify=y
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=7, stratify=y_dev
)
scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)
readout = kron_all([I, Z])


def predict_scores(
    X: np.ndarray, params: np.ndarray
) -> np.ndarray:
    """Return target-qubit Z scores for scaled input rows."""
    return np.array([vqc_score(x, params, readout) for x in X])


def loss_gradient(X, y, params):
    """Return the MSE gradient for signed targets y."""
    residual = predict_scores(X, params) - y
    gradient = np.zeros_like(params)
    for j in range(len(params)):
        plus, minus = params.copy(), params.copy()
        plus[j] += np.pi / 2
        minus[j] -= np.pi / 2
        derivative = 0.5 * (
            predict_scores(X, plus) - predict_scores(X, minus)
        )
        gradient[j] = 2 * np.mean(residual * derivative)
    return gradient


params = np.random.default_rng(7).normal(0.0, 0.3, size=4)
best_val_loss = np.inf
best_params = params.copy()
for epoch in range(121):
    train_scores = predict_scores(X_train, params)
    val_scores = predict_scores(X_val, params)
    train_loss = np.mean((train_scores - y_train) ** 2)
    val_loss = np.mean((val_scores - y_val) ** 2)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = params.copy()
        best_epoch = epoch
    if epoch % 40 == 0:
        print(f"epoch={epoch:3d} train={train_loss:.4f} "
              f"val={val_loss:.4f}")
    if epoch < 120:
        params -= 0.2 * loss_gradient(X_train, y_train, params)

val_scores = predict_scores(X_val, best_params)
test_scores = predict_scores(X_test, best_params)
val_predictions = np.where(val_scores >= 0, 1, -1)
test_predictions = np.where(test_scores >= 0, 1, -1)
margins = y_val * val_scores
val_accuracy = np.mean(val_predictions == y_val)
test_accuracy = np.mean(test_predictions == y_test)
print("selected epoch:", best_epoch)
print(f"validation accuracy: {val_accuracy:.3f}")
print(f"test accuracy: {test_accuracy:.3f}")
print(f"validation margin: min={margins.min():.3f}, "
      f"median={np.median(margins):.3f}")


In [ ]:
import matplotlib.pyplot as plt

plot_X = np.vstack([X_train_raw, X_val_raw])
lengths = np.linspace(plot_X[:, 0].min() - 0.2,
                      plot_X[:, 0].max() + 0.2, 100)
widths = np.linspace(plot_X[:, 1].min() - 0.2,
                     plot_X[:, 1].max() + 0.2, 100)
xx, yy = np.meshgrid(lengths, widths)
grid_raw = np.column_stack([xx.ravel(), yy.ravel()])
surface = predict_scores(scaler.transform(grid_raw), best_params)
surface = surface.reshape(xx.shape)
fig, ax = plt.subplots()
contours = ax.contourf(
    xx, yy, surface, levels=np.linspace(-1, 1, 21),
    cmap="coolwarm", vmin=-1, vmax=1
)
ax.contour(xx, yy, surface, levels=[0], colors="black")
ax.scatter(X_val_raw[:, 0], X_val_raw[:, 1], c=y_val,
           cmap="coolwarm", vmin=-1, vmax=1, edgecolors="black")
ax.set(xlabel="Petal length (cm)", ylabel="Petal width (cm)")
ax.set_title("Validation: setosa (-1), versicolor (+1)")
fig.colorbar(contours, ax=ax, label="VQC score")
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
I = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
CNOT_01 = np.array(
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ],
    dtype=complex,
)
rng = np.random.default_rng(7)


def kron_all(ops):
    output = ops[0]
    for op in ops[1:]:
        output = np.kron(output, op)
    return output


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2),  np.cos(theta / 2)],
        ],
        dtype=complex,
    )


def rz(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.exp(-0.5j * theta), 0.0],
            [0.0, np.exp(0.5j * theta)],
        ],
        dtype=complex,
    )


def trainable_layer(params: np.ndarray) -> np.ndarray:
    theta0, theta1, theta2, theta3 = params
    return CNOT_01 @ kron_all([ry(theta0), ry(theta1)]) @ kron_all(
        [rz(theta2), rz(theta3)]
    )


def score(x: np.ndarray, params_by_layer: np.ndarray) -> float:
    state = kron_all([ket0, ket0])
    state = kron_all([ry(x[0]), ry(x[1])]) @ state

    unitary = np.eye(4, dtype=complex)
    for params in params_by_layer:
        unitary = trainable_layer(params) @ unitary

    state = unitary @ state
    observable = kron_all([Z, I])
    return float(np.vdot(state, observable @ state).real)


def mean_abs_gradient(
    x: np.ndarray, params_by_layer: np.ndarray
) -> float:
    flat = params_by_layer.reshape(-1)
    grads = []
    for index in range(len(flat)):
        shifted_plus = flat.copy()
        shifted_minus = flat.copy()
        shifted_plus[index] += np.pi / 2
        shifted_minus[index] -= np.pi / 2

        plus = score(x, shifted_plus.reshape(params_by_layer.shape))
        minus = score(
            x, shifted_minus.reshape(params_by_layer.shape)
        )
        grads.append(abs(0.5 * (plus - minus)))

    return float(np.mean(grads))


x = np.array([0.4, -0.7])
for depth in [1, 2, 4, 6]:
    sampled = []
    for _ in range(32):
        params = rng.uniform(0.0, 2.0 * np.pi, size=(depth, 4))
        sampled.append(mean_abs_gradient(x, params))

    print(f"depth={depth:>2d} mean|grad|={np.mean(sampled):.6f}")
